# Read the Bronze Data and create Frame


#### Some important notes


In [ ]:

# 🔥 PySpark All-in-One Cheat Sheet (With Real Examples)

from pyspark.sql.functions import *

# Assume dataset columns:
# customer_id, name, email, age, city, gender, dob, sales

# ---------------- BASIC ----------------
df.show()                                  # display data

df.printSchema()                           # check schema
df.columns                                 # list columns
df.count()                                 # total rows

# ---------------- SELECT / FILTER ----------------

df.select("customer_id", "email")           # select specific columns
df.filter(col("age") > 18)                  # filter adults
df.where("city = 'pune'")                   # filter by city

# ---------------- ADD / MODIFY COLUMN ----------------

df.withColumn("age_plus_1", col("age") + 1)    # increase age by 1
df.withColumn("email_clean", lower(trim(col("email"))))   # clean email

# ---------------- RENAME / DROP ----------------

df.withColumnRenamed("dob", "date_of_birth")   # rename column
df.drop("age")                                 # remove column

# ---------------- NULL HANDLING ----------------

df.dropna(subset=["customer_id"])              # remove rows with null id
df.fillna({"city": "Unknown"})                 # fill null city

# ---------------- REMOVE DUPLICATES ----------------

df.dropDuplicates(["customer_id"])             # unique customers

# ---------------- STRING CLEANING ----------------

df.withColumn("name", initcap(trim(col("name"))))   # clean name
df.withColumn("city", upper(col("city")))           # uppercase city

# ---------------- REPLACE VALUES ----------------

df.replace({"NA": None, "": None}, subset=["email"])   # fix bad values

# ---------------- DATE HANDLING ----------------

df.withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))   # clean date

# ---------------- CONDITIONAL LOGIC ----------------

df.withColumn(
"gender_clean",
when(lower(col("gender")) == "m", "Male")
.when(lower(col("gender")) == "f", "Female")
.otherwise("Other")
)

# ---------------- AGGREGATION ----------------

df.groupBy("city").count()                     # customers per city
df.groupBy("city").agg(sum("sales"))           # total sales per city

# ---------------- SORTING ----------------

df.orderBy("age")                              # sort by age
df.orderBy(col("sales").desc())                # highest sales first

# ---------------- JOINS ----------------

orders = spark.table("orders")

df.join(orders, "customer_id", "inner")        # combine customer + orders

# ---------------- READ DATA ----------------

spark.read.csv("Files/bronze/customers.csv", header=True, inferSchema=True)

# ---------------- WRITE DATA ----------------

df.write.format("delta").saveAsTable("silver_customers")
df.write.mode("overwrite").parquet("Files/silver/customers")

# ---------------- EXPRESSION ----------------

df.withColumn("double_sales", expr("sales * 2"))   # calculate new column

# ---------------- 🔥 REAL CLEANING PATTERN ----------------

df_clean = (
df
.withColumn("email", lower(trim(col("email"))))     # clean email
.withColumn("name", initcap(trim(col("name"))))     # clean name
.dropna(subset=["customer_id"])                     # remove null ids
.dropDuplicates(["customer_id"])                    # remove duplicates
)

In [1]:
customer_raw=spark.read.parquet("abfss://Ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/customers.csv")
orders_raw=spark.read.parquet("abfss://Ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/orders.csv")
payment_raw=spark.read.parquet("abfss://Ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/payments.csv")
support_ticket_raw=spark.read.parquet("abfss://Ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/support_tickets.csv")
web_activities_raw=spark.read.parquet("abfss://Ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/web_activities.csv")



StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 3, Finished, Available, Finished, False)

#### Customer Data frame

In [3]:
display(customer_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8f1893fa-9b1e-4425-b587-6955d96d1575)

In [7]:
display(orders_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 435bd1a9-9aeb-4723-947c-d5e3a55469e5)

In [6]:
display(payment_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 525ab65e-b56c-42f5-bf72-77c432ec2839)

In [4]:
display(support_ticket_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77fc63cd-65a1-4030-aed4-14dbc01beb8e)

In [5]:
display(web_activities_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9f94b30-48af-4433-b195-7a5dbb3a5dc6)

### Create Delta Bronze table

In [9]:
# Save as Bronze Delta Tables
customer_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payment_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_ticket_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_activities_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 11, Finished, Available, Finished, False)

#### Clean the data Silver

##### Customer Data clean


In [10]:
display(customer_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a22c3580-ade1-4447-aa5f-d8d7d153c71f)

In [12]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# cleaning raw customer data 
customers_clean = (
    customer_raw
    
    # fix email → remove spaces + make lowercase
    .withColumn("email", lower(trim(col("EMAIL"))))
    
    # clean name → remove extra spaces + proper format
    .withColumn("name", initcap(trim(col("name"))))
    
    # handle gender values → keep it consistent
    .withColumn(
        "gender",
        when(lower(col("gender")).isin("f", "female"), "Female")
        .when(lower(col("gender")).isin("m", "male"), "Male")
        .otherwise("Other")
    )
    
    # replace '/' with '-' then convert to date
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    
    # clean location → make it readable
    .withColumn("location", initcap(col("location")))
    
    # remove duplicate customer records
    .dropDuplicates(["customer_id"])
    
    # drop rows where key fields are missing
    .dropna(subset=["customer_id", "email"])
)

display(customers_clean.limit(6))

# save cleaned data into silver layer
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, deec649c-5795-4f92-b212-8fd599309b84)

##### Order Data clean

In [14]:
display(orders_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a701be6f-9f71-44d4-9cde-db490da656a4)

In [15]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1857ec20-76e3-4147-b7a5-9d99bd5b7e1f)

##### Payment data clean

In [16]:
display(payment_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1b134188-553f-4a93-9397-814a86e26cdf)

In [18]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))
payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")


StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff14a806-4c87-4e38-b3af-277493ba4b72)

##### Support data clean

In [19]:
%%sql
SELECT * from support lii

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 21, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [ ]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

##### Web data clean

In [21]:
display(web_activities_raw.limit(5))

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c5ddc659-cf2c-40e7-b1fa-bfd2500db6c6)

In [22]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b4ef2b0-9cec-4acf-8b63-e94ec984b08f)

## GOLD Tables aggregate

In [24]:
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 75be5523-2fcf-4f29-bbd1-50b7d748355b, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5dc2dca4-c567-4fc8-a9de-190985c24c98)